In [ ]:
import logging
import os
import numpy as np
import open3d as o3d
from PIL import Image
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

from marmopose.version import __version__ as marmopose_version
from marmopose.config import Config

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(name)s - %(message)s')
logger = logging.getLogger(__name__)

logger.info(f'MarmoPose version: {marmopose_version}')

from marmopose.utils.data_io import load_points_3d_h5


In [ ]:
os.chdir('..')
config_path = '../configs/default.yaml'

config = Config(
    config_path=config_path,
    
    n_tracks=1,
    project='../demos/single',
)
os.chdir('T-SNE_Watershed')


In [ ]:
SRC_DIR = "/scratch/VideoTracking/Videos/Test3.5"
points_3d = load_points_3d_h5(os.path.join(SRC_DIR,"Output/points_3d/optimized.h5"))
idx_spinemid = config.animal['bodyparts'].index('spinemid')
idx_tailbase = config.animal['bodyparts'].index('tailbase')
idx_neck = config.animal['bodyparts'].index('neck')
points_3d_ = points_3d - points_3d[:,:,[idx_spinemid],:]
vector_body = points_3d_[:,:,[idx_neck],:] - points_3d_[:,:,[idx_tailbase],:]
magnitudes_vector_body_xy = np.sqrt(vector_body[0,:,0,0]**2 + vector_body[0,:,0, 1]**2)[:,np.newaxis,np.newaxis]
rotation_matrices = np.array([[vector_body[0,:,0,0],vector_body[0,:,0,1]],[-vector_body[0,:,0,1],vector_body[0,:,0,0]]]).transpose((2,0,1))/magnitudes_vector_body_xy
rotated_points_3d = np.copy(points_3d_)[0,:,:,:]
rotated_points_3d[:,:,:2] = np.einsum('ikl,ijl -> ijk', rotation_matrices, rotated_points_3d[:,:,:2])
rotated_points_3d_ = rotated_points_3d.reshape((-1,48))
rotated_points_3d_velocity = rotated_points_3d_[1:,:] - rotated_points_3d_[:-1,:]
rotated_points_3d_velocity_3d = rotated_points_3d_velocity.reshape(-1,16,3)
inputs = np.concatenate((rotated_points_3d_[:-1,:],rotated_points_3d_velocity), axis = 1)

In [ ]:
scaler = StandardScaler()
normalized_inputs = scaler.fit_transform(inputs)
pca = PCA(n_components=0.95)
inputs_pca = pca.fit_transform(normalized_inputs)
print(f"Number of principal components retained: {inputs_pca.shape[1]}")
print(pca.explained_variance_ratio_)

In [ ]:
%matplotlib inline
plt.clf()
tsne = TSNE(n_components=2, perplexity=50, learning_rate=200, n_iter=5000, random_state=0)
X_tsne = tsne.fit_transform(inputs_pca)
fig = plt.figure()
ax = fig.add_subplot(111)
ax.scatter(X_tsne[:,0],X_tsne[:,1])
fig.show()


# for i in range(10,81,10):
#     for n in range(1000,5001,1000):
#         tsne = TSNE(n_components=2, perplexity=i, learning_rate=200, n_iter=n, random_state=42)
#         X_tsne = tsne.fit_transform(inputs_pca)
#         fig = plt.figure()
#         ax = fig.add_subplot(111)
#         ax.scatter(X_tsne[:,0],X_tsne[:,1])
#         ax.set_title(f'Perpexity {i} N_iter {n} Divergence {tsne.kl_divergence_}')
#         # ax.scatter(X_tsne[:,0],X_tsne[:,1],X_tsne[:,2])
#         fig.show()

# Choice: Perplexity 30 and 5000 iterations
# np.random.seed(3)
# rss = np.random.randint(0,100,10)
# for rs in rss:
#     tsne = TSNE(n_components=2, perplexity=30, learning_rate=200, n_iter=5000, random_state=rs)
#     X_tsne = tsne.fit_transform(inputs_pca)
#     fig = plt.figure()
#     ax = fig.add_subplot(111)
#     ax.scatter(X_tsne[:,0],X_tsne[:,1])
#     ax.set_title(f'Perpexity {i} N_iter {n} Divergence {tsne.kl_divergence_}')
#     # ax.scatter(X_tsne[:,0],X_tsne[:,1],X_tsne[:,2])
#     fig.show()


In [ ]:
%matplotlib inline
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

wcss = []
for i in range(1, 11):
    kmeans = KMeans(n_clusters=i, init='k-means++', random_state=42)
    kmeans.fit(X_tsne)
    wcss.append(kmeans.inertia_)

# Plot the Elbow Method graph
plt.plot(range(1, 11), wcss, marker='o')
plt.title('Elbow Method')
plt.xlabel('Number of clusters')
plt.ylabel('WCSS')
plt.show()


In [ ]:
from sklearn.metrics import silhouette_score

silhouette_scores = []
for i in range(2, 31):
    kmeans = KMeans(n_clusters=i, init='k-means++', random_state=42)
    cluster_labels = kmeans.fit_predict(X_tsne)
    silhouette_avg = silhouette_score(X_tsne, cluster_labels)
    silhouette_scores.append(silhouette_avg)
    print(f"For {i} clusters, the silhouette score is: {silhouette_avg}")

n_clusters_opt = np.argmax(silhouette_scores) + 2

plt.plot(range(2, 31), silhouette_scores, marker='o')
plt.title('Silhouette Scores')
plt.xlabel('Number of clusters')
plt.ylabel('Silhouette Score')
plt.show()


In [ ]:
kmeans = KMeans(n_clusters=n_clusters_opt, init='k-means++', random_state=42)
clusters = kmeans.fit_predict(X_tsne)


In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
np.random.seed(3)
# Create a 3D scatter plot
fig = plt.figure()
ax = fig.add_subplot(111)
c = np.random.rand(n_clusters_opt,3)

ax.scatter(X_tsne[:,0],X_tsne[:,1], c=c[clusters])
fig.show()

In [ ]:
np.random.seed(3)
idxs = np.array([np.random.choice(np.nonzero(clusters == i)[0],size = 5) for i in range(n_clusters_opt)])

In [ ]:
import cv2
frames = np.empty((4,*idxs.shape, 1080, 1920, 3))
for i in range(1,5):
  vidcap = cv2.VideoCapture(os.path.join(SRC_DIR,f'Output/videos_labeled_2d/output{i}.mp4'))
  success,image = vidcap.read()
  j = 0
  while success:
    if not (j % 500):
      print(f'Video {i} frame {j}')
    id_idxs = np.nonzero(idxs == j)
    if id_idxs[0].size:
      image[..., 0:3] = image[..., ::-1]
      frames[i - 1, id_idxs[0][0], id_idxs[1][0]] = image
    success,image = vidcap.read()
    j += 1
  id_idxs = np.nonzero(idxs == j)
  if id_idxs[0].size:
    image[..., 0:3] = image[..., ::-1]
    frames[i - 1, id_idxs[0][0], id_idxs[1][0]] = image


In [ ]:
%matplotlib inline
rotated_points_3d_velocity_3d_ = rotated_points_3d_velocity_3d.reshape(-1,16,3)
import matplotlib.colors as mcolors
for i in range(5):
# for i in range(n_clusters_opt):
    for j in range(5):
        fig, ax = plt.subplots(figsize=(16, 4))
        ax.axis('off')
        norm = mcolors.Normalize(vmin=-20, vmax=20)
        data = rotated_points_3d_velocity_3d_[idxs[i,j],:,:].transpose((1,0))
        cmap = plt.cm.RdYlBu
        table = ax.table(
            cellText=np.round(data,decimals=2),
            cellColours= cmap(norm(data)),
            rowLabels=['Velocity X','Velocity Y','Velocity Z'],
            colLabels=config.animal['bodyparts'],
            loc='center',
            cellLoc='center',
            fontsize = 20
        )
        plt.show()



        fig, ax = plt.subplots(2,2,figsize = (30,15))
        for k in range(4):
            frame = frames[k,i,j]/255
            ax[int(k/2)][k%2].imshow(frame)
            ax[int(k/2)][k%2].set_title(f'Cluster {i}, frame {idxs[i,j]}, camera {k}')
            ax[int(k/2)][k%2].axis('off')
        fig.show()

In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

xlim, ylim, zlim, _ = config.visualization['room_dimensions']

for i in range(n_clusters_opt):
    for j in range(5):
        # Create a 3D scatter plot
        fig = plt.figure()
        ax = fig.add_subplot(111, projection='3d')
        for bodyparts in config.visualization['skeleton'][::-1]:
            idx_bodyparts = []
            for bodypart in bodyparts:
                idx_bodyparts.append(config.animal['bodyparts'].index(bodypart))
            ax.plot(points_3d[0,idxs[i,j],idx_bodyparts,0],points_3d[0,idxs[i,j],idx_bodyparts,1],points_3d[0,idxs[i,j],idx_bodyparts,2], marker = 'o', ms=3)
        ax.set_title(f'Cluster {i}, frame {idxs[i,j]}')
        ax.set_xlim((0,xlim))
        ax.set_ylim((0,ylim))
        ax.set_zlim((0,zlim))

In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

xlim, ylim, zlim, _ = config.visualization['room_dimensions']

for i in range(n_clusters_opt):
    for j in range(5):
        # Create a 3D scatter plot
        fig = plt.figure()
        ax = fig.add_subplot(111, projection='3d')
        for bodyparts in config.visualization['skeleton'][::-1]:
            idx_bodyparts = []
            for bodypart in bodyparts:
                idx_bodyparts.append(config.animal['bodyparts'].index(bodypart))
            ax.plot(rotated_points_3d[idxs[i,j],idx_bodyparts,0],rotated_points_3d[idxs[i,j],idx_bodyparts,1],rotated_points_3d[idxs[i,j],idx_bodyparts,2], marker = 'o', ms=3)
        ax.set_title(f'Cluster {i}, frame {idxs[i,j]}')
        # ax.set_xlim((0,xlim))
        # ax.set_ylim((0,ylim))
        # ax.set_zlim((0,zlim))